In [1]:
import numpy as np
import pandas as pd
import os

print("Libraries imported successfully!")
print(f"Numpy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

Libraries imported successfully!
Numpy version: 1.26.4
Pandas version: 2.2.2


In [2]:
np.random.seed(42)  # makes results reproducible 
n = 2000            # 2000 deliveries

# Generating each column
distance        = np.random.randint(50, 800, n)
weather         = np.random.randint(1, 6, n)
road_type       = np.random.choice(['highway', 'rural'], n, p=[0.6, 0.4])
season          = np.random.choice(['winter','spring','summer','fall'], n)
cargo_weight    = np.random.randint(500, 20000, n)
driver_exp      = np.random.randint(0, 30, n)
departure_hour  = np.random.randint(4, 22, n)

print("Features generated!")
print(f"Total deliveries: {n}")
print(f"Sample distances: {distance[:5]}")
print(f"Sample weather:   {weather[:5]}")
print(f"Sample seasons:   {season[:5]}")

Features generated!
Total deliveries: 2000
Sample distances: [152 485 320 156 121]
Sample weather:   [3 2 3 2 5]
Sample seasons:   ['summer' 'spring' 'spring' 'spring' 'summer']


In [3]:
# This is the logic that decides which deliveries get delayed
# Each condition adds to a "delay score"
# Higher score = more likely to be delayed

delay_score = (
    (weather >= 4).astype(int)        * 3 +   # bad weather = biggest risk
    (distance > 500).astype(int)      * 2 +   # long distance = high risk
    (road_type == 'rural').astype(int)* 2 +   # rural roads = high risk
    (season == 'winter').astype(int)  * 2 +   # winter = high risk
    (cargo_weight > 15000).astype(int)* 1 +   # heavy cargo = slight risk
    (driver_exp < 2).astype(int)      * 2 +   # new driver = high risk
    (departure_hour > 18).astype(int) * 1     # late departure = slight risk
)

# Converting score into a probability and then randomly assigning 0 or 1
delay_probability = delay_score / delay_score.max()
is_delayed = (np.random.rand(n) < delay_probability).astype(int)

print(f"Total delayed:  {is_delayed.sum()}")
print(f"Total on time:  {(is_delayed == 0).sum()}")
print(f"Delay rate:     {is_delayed.mean():.1%}")

Total delayed:  726
Total on time:  1274
Delay rate:     36.3%


In [4]:
# Compiling all columns together into one DataFrame
df = pd.DataFrame({
    'distance_km':          distance,
    'weather_severity':     weather,
    'road_type':            road_type,
    'season':               season,
    'cargo_weight_kg':      cargo_weight,
    'driver_experience_yrs':driver_exp,
    'departure_hour':       departure_hour,
    'is_delayed':           is_delayed
})

# Save to CSV
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/deliveries.csv', index=False)

print("Dataset saved to data/raw/deliveries.csv")
print(f"\nShape: {df.shape}")
print(f"\nFirst 5 rows:")
df.head()

Dataset saved to data/raw/deliveries.csv

Shape: (2000, 8)

First 5 rows:


,distance_km,weather_severity,road_type,season,cargo_weight_kg,driver_experience_yrs,departure_hour,is_delayed
0,152,3,highway,summer,18312,16,6,0
1,485,2,highway,spring,4878,17,9,0
2,320,3,highway,spring,8029,0,11,0
3,156,2,rural,spring,3571,18,17,0
4,121,5,rural,summer,17439,21,13,1
